# Image Captioning — Cross-Modal Generation

An image captioning model built from scratch with PyTorch. A ResNet18 encoder produces a grid of spatial image features; a transformer decoder cross-attends to those features and autoregressively generates a caption, one token at a time.

```
Image (224×224)                Caption tokens (shifted right)
      │                                       │
[ResNet18 → 7×7×512]              [Token + position embeddings]
      │                                       │
[1×1 conv → d_model]                          │
      │                                       │
  49 image "tokens" ────cross-attention────> [Transformer Decoder × N]
                                              │
                                       [Linear → vocab logits]
                                              │
                                       next-token prediction
```

In [ ]:
!pip install torch torchvision transformers kagglehub matplotlib

## Download Dataset
Downloads Flickr8k from Kaggle via `kagglehub`. Requires a one-time Kaggle setup:
1. Go to kaggle.com → Settings → API → Create New Token
2. Upload the downloaded `kaggle.json` to `~/.kaggle/kaggle.json`

In [ ]:
import os
import shutil
import kagglehub

DATASET_DIR = "dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "Images")
CAPTIONS_FILE = os.path.join(DATASET_DIR, "captions.txt")

if os.path.isdir(IMAGES_DIR) and os.path.isfile(CAPTIONS_FILE):
    print("Dataset already exists, skipping download.")
else:
    print("Downloading Flickr8k from Kaggle...")
    path = kagglehub.dataset_download("adityajn105/flickr8k")
    print(f"Downloaded to: {path}")

    os.makedirs(DATASET_DIR, exist_ok=True)
    src_images = os.path.join(path, "Images")
    src_captions = os.path.join(path, "captions.txt")

    if not os.path.isdir(IMAGES_DIR):
        shutil.copytree(src_images, IMAGES_DIR)
    if not os.path.isfile(CAPTIONS_FILE):
        shutil.copy2(src_captions, CAPTIONS_FILE)

    print(f"Dataset ready: {len(os.listdir(IMAGES_DIR))} images, captions.txt present.")

In [ ]:
import csv
import math
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from transformers import BertTokenizer

In [ ]:
# --------------- Config ---------------
CAPTIONS_FILE = "dataset/captions.txt"
IMAGES_DIR = "dataset/Images"
CHECKPOINT_DIR = "checkpoints"

EPOCHS = 20
LR = 3e-4
LOG_EVERY = 50
VAL_SPLIT = 0.1
BATCH_SIZE = 64
MAX_TOKEN_LENGTH = 32

# Decoder hyperparameters
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
FFN_DIM = 1024
DROPOUT = 0.1

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
# Pin to GPU 1 on the shared JupyterHub (GPU 0 is usually saturated).
# Must run BEFORE any torch.cuda call — if CUDA was already touched this session,
# restart the kernel after editing this line.
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

if torch.cuda.is_available():
    device = "cuda"
    free, total = torch.cuda.mem_get_info(0)
    print(f"Device: {device} ({torch.cuda.get_device_name(0)}) — {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
elif torch.backends.mps.is_available():
    device = "mps"
    print(f"Device: {device}")
else:
    device = "cpu"
    print(f"Device: {device}")

## Data Pipeline

Each sample yields `(image_tensor, input_ids)`. The `input_ids` already include BERT's `[CLS]` (which we treat as BOS) and `[SEP]` (which we treat as EOS), padded to `MAX_TOKEN_LENGTH` with `[PAD]` (id 0).

In [ ]:
class Flickr8kDataset(Dataset):
    def __init__(self, captions_file: str, images_dir: str):
        self.images_dir = images_dir

        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

        self.tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
        # Reuse BERT's special tokens for the captioning task
        self.bos_id = self.tokenizer.cls_token_id   # 101 — sequence-start marker
        self.eos_id = self.tokenizer.sep_token_id   # 102 — sequence-end marker
        self.pad_id = self.tokenizer.pad_token_id   #   0 — ignored by loss + attention
        self.vocab_size = self.tokenizer.vocab_size

        self.samples = []  # list of (image_filename, caption)
        with open(captions_file, "r") as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row in reader:
                self.samples.append((row[0], row[1]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_filename, caption = self.samples[idx]
        image_path = os.path.join(self.images_dir, image_filename)

        image = Image.open(image_path).convert("RGB")
        image = self.image_transform(image)

        # Tokenizer prepends [CLS] and appends [SEP] automatically.
        tokens = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=MAX_TOKEN_LENGTH,
            return_tensors="pt",
        )
        input_ids = tokens["input_ids"].squeeze(0)  # (MAX_TOKEN_LENGTH,)
        return image, input_ids

In [ ]:
dataset = Flickr8kDataset(CAPTIONS_FILE, IMAGES_DIR)
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)
print(f"Train: {train_size}  Val: {val_size}")
print(f"Vocab size: {dataset.vocab_size}  PAD={dataset.pad_id}  BOS={dataset.bos_id}  EOS={dataset.eos_id}")

num_workers = 4 if torch.cuda.is_available() else 2
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=num_workers, pin_memory=True)

## Image Encoder

ResNet18 truncated before its global pooling layer, so the output keeps **spatial structure**: a 7×7 grid of 512-d feature vectors. We project each cell to `D_MODEL` with a 1×1 conv and flatten the grid into a sequence of 49 "image tokens" that the decoder can cross-attend to.

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, d_model: int = D_MODEL):
        super().__init__()
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT)
        # Drop avgpool + fc — keep through layer4. Output: (B, 512, 7, 7)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.projection = nn.Conv2d(512, d_model, kernel_size=1)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(images)            # (B, 512, 7, 7)
        feat = self.projection(feat)             # (B, d_model, 7, 7)
        # Flatten spatial dims and put sequence axis second: (B, 49, d_model)
        return feat.flatten(2).transpose(1, 2)

## Caption Decoder

A standard transformer decoder. Each block has three sub-layers:
- **Causal self-attention** — each token attends to itself and earlier tokens, never future ones (the *causal mask* makes future positions invisible).
- **Cross-attention** — each decoder position attends to the 49 image tokens, letting the language model "see" the image.
- **Feed-forward** — a per-token MLP.

The output projection shares weights with the input token embedding (*weight tying*), which both halves the parameter count and tends to improve generalization.

In [ ]:
class CaptionDecoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        d_model: int = D_MODEL,
        n_heads: int = N_HEADS,
        n_layers: int = N_LAYERS,
        ffn_dim: int = FFN_DIM,
        dropout: float = DROPOUT,
        max_len: int = MAX_TOKEN_LENGTH,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

        # Output head shares weights with the input embedding (weight tying)
        self.output = nn.Linear(d_model, vocab_size, bias=False)
        self.output.weight = self.token_embedding.weight

    def forward(self, input_ids: torch.Tensor, image_features: torch.Tensor) -> torch.Tensor:
        # input_ids: (B, T)   image_features: (B, 49, d_model)
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.dropout(x)

        # Causal mask: True where attention is *blocked* (upper triangle, excluding diagonal)
        causal_mask = torch.triu(
            torch.ones(T, T, device=input_ids.device, dtype=torch.bool), diagonal=1
        )
        # Padding mask: True where token == PAD — those positions are ignored by attention
        padding_mask = (input_ids == self.pad_id)

        out = self.decoder(
            tgt=x,
            memory=image_features,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=padding_mask,
        )
        out = self.norm(out)
        return self.output(out)  # (B, T, vocab_size)

## Captioning Model & Optimizer

In [ ]:
class CaptioningModel(nn.Module):
    def __init__(self, vocab_size: int, pad_id: int):
        super().__init__()
        self.encoder = ImageEncoder()
        self.decoder = CaptionDecoder(vocab_size=vocab_size, pad_id=pad_id)

    def forward(self, images: torch.Tensor, input_ids: torch.Tensor) -> torch.Tensor:
        image_features = self.encoder(images)
        return self.decoder(input_ids, image_features)


model = CaptioningModel(vocab_size=dataset.vocab_size, pad_id=dataset.pad_id).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

## Training Functions

**Teacher forcing** — during training, we feed the ground-truth previous tokens (not the model's own predictions). This lets every position be trained in parallel: the decoder sees `[BOS, w1, w2, …, w_{n-1}]` and is asked to predict `[w1, w2, …, EOS]`.

We report **perplexity** (PPL = exp(loss)) alongside loss — it's the more interpretable metric for language models: "on average the model is choosing among PPL plausible next words."

In [ ]:
def caption_loss(logits: torch.Tensor, targets: torch.Tensor, pad_id: int) -> torch.Tensor:
    # logits: (B, T, V)   targets: (B, T)
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=pad_id,
    )


def train_one_epoch(model, optimizer, dataloader, device, epoch, pad_id):
    model.train()
    running_loss = 0.0
    for step, (images, input_ids) in enumerate(dataloader):
        images = images.to(device)
        input_ids = input_ids.to(device)

        # Teacher forcing: drop last token from input, drop first token from target
        decoder_input = input_ids[:, :-1]
        targets = input_ids[:, 1:]

        logits = model(images, decoder_input)
        loss = caption_loss(logits, targets, pad_id)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        if (step + 1) % LOG_EVERY == 0:
            avg = running_loss / LOG_EVERY
            ppl = math.exp(min(avg, 20))
            print(f"  Epoch {epoch+1} | Step {step+1:>4d} | Loss {avg:.4f} | PPL {ppl:.2f}")
            running_loss = 0.0


@torch.no_grad()
def validate(model, dataloader, device, pad_id):
    model.eval()
    total_loss, num_batches = 0.0, 0
    for images, input_ids in dataloader:
        images = images.to(device)
        input_ids = input_ids.to(device)
        decoder_input = input_ids[:, :-1]
        targets = input_ids[:, 1:]
        logits = model(images, decoder_input)
        loss = caption_loss(logits, targets, pad_id)
        total_loss += loss.item()
        num_batches += 1
    return total_loss / num_batches

## Training Loop

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    t0 = time.time()
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"{'='*50}")

    train_one_epoch(model, optimizer, train_loader, device, epoch, dataset.pad_id)

    val_loss = validate(model, val_loader, device, dataset.pad_id)
    val_ppl = math.exp(min(val_loss, 20))
    elapsed = time.time() - t0
    print(f"  Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f} | Time: {elapsed:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        path = os.path.join(CHECKPOINT_DIR, "best_caption_model.pt")
        torch.save({
            "epoch": epoch + 1,
            "val_loss": val_loss,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
        }, path)
        print(f"  ** New best model saved (val_loss={val_loss:.4f}) **")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## Caption Generation

**Greedy decoding**: start with `[CLS]`, at each step take the token with the highest probability, append it to the sequence, and stop on `[SEP]` or when we hit the max length. Greedy is the simplest decoder — deterministic and fast, though beam search usually produces slightly better captions.

In [ ]:
@torch.no_grad()
def generate_caption(model, image: torch.Tensor, dataset, device, max_len: int = MAX_TOKEN_LENGTH) -> str:
    model.eval()
    image = image.unsqueeze(0).to(device)
    image_features = model.encoder(image)  # (1, 49, d_model) — computed once, reused every step

    tokens = [dataset.bos_id]
    for _ in range(max_len - 1):
        input_ids = torch.tensor([tokens], device=device)
        logits = model.decoder(input_ids, image_features)   # (1, T, V)
        next_id = logits[0, -1].argmax().item()              # greedy pick
        if next_id == dataset.eos_id:
            break
        tokens.append(next_id)

    # Drop the leading BOS for display
    return dataset.tokenizer.decode(tokens[1:], skip_special_tokens=True)

## Demo — Generated Captions vs. Ground Truth

For six unique images from the validation set, generate a caption and display it next to the human-written reference.

In [ ]:
import matplotlib.pyplot as plt

# Load best checkpoint
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best_caption_model.pt"),
                  map_location=device, weights_only=True)
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded checkpoint — epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

# Pick 6 unique images from validation
seen = set()
demo_indices = []
for i in range(len(val_dataset)):
    real_idx = val_dataset.indices[i]
    img_filename = val_dataset.dataset.samples[real_idx][0]
    if img_filename not in seen:
        seen.add(img_filename)
        demo_indices.append(real_idx)
    if len(demo_indices) == 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle("Generated Captions vs. Ground Truth", fontsize=16, fontweight="bold")

for ax, idx in zip(axes.flat, demo_indices):
    image, input_ids = dataset[idx]
    truth = dataset.tokenizer.decode(input_ids, skip_special_tokens=True)
    generated = generate_caption(model, image, dataset, device)

    img_filename = dataset.samples[idx][0]
    img_pil = Image.open(os.path.join(IMAGES_DIR, img_filename)).convert("RGB")
    ax.imshow(img_pil)
    ax.set_title(f'GEN: "{generated}"\nGT:  "{truth}"', fontsize=11, loc="left", pad=8)
    ax.axis("off")

plt.tight_layout()
os.makedirs("assets", exist_ok=True)
plt.savefig("assets/captioning_demo.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → assets/captioning_demo.png")